# Sample CloudSql Connection

# Need to set up for each instance - Add your VM's IP to the Authorized Networks
**1. Find the external IP of your JupyterLab VM**
https://console.cloud.google.com/compute/instances?project=adsp-34002-on02-sopho-scribe&authuser=1
 * Go to VM Instances
 * Find your JupyterLab VM
 * Copy the External IP address (looks like 34.91.100.45).

**2. Add the VM's IP to your Cloud SQL authorized networks**
https://console.cloud.google.com/sql/instances/currensee-sql/connections/networking?authuser=1&project=adsp-34002-on02-sopho-scribe
* Go to Cloud SQL instances. 
* Click your instance.
* Click Connections in the left sidebar.
* Scroll to Authorized networks → Add network.
* Name: anything like jupyterlab-vm
* Network: paste the external IP you just copied (e.g., 34.91.100.45/32)
* IMPORTANT: Add /32 to allow only that single IP.
* Click Save.

It will take ~30 seconds to update.

In [1]:
from google.cloud import secretmanager
import pandas as pd
from currensee.utils.db_utils import create_pg_engine

## IMPORTANT: .env Configuration

In order for the DB credentials to be loaded properly, you will need to have a file located at 

`<fl>_currensee/currensee/.env`

with the following contents:

```bash
GOOGLE_API_KEY


```



## Purpose of this notebook

This notebook is designed to load data into the crm_outlook database by copying static data from the outlook database (loaded via load_outlook_to_crm) and dynamic data from the crm database (generated by load_crm_to_cloud). If any changes have been made in these databases, ensure that the existing data in crm_outlook is updated accordingly.

In [2]:
from currensee.core.settings import Settings
settings = Settings()

In [3]:
import inspect
print(inspect.signature(create_pg_engine))

(db_name: str)


In [4]:
def get_table_and_column_info(db_name):
    engine = create_pg_engine(db_name)

    tables_query = """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public' AND table_type = 'BASE TABLE';
    """
    tables = pd.read_sql(tables_query, engine)

    columns_query = """
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public';
    """
    columns = pd.read_sql(columns_query, engine)

    return tables, columns

In [5]:
crm_tables, crm_columns = get_table_and_column_info("crm")
outlook_tables, outlook_columns = get_table_and_column_info("outlook")

print("CRM Tables:")
for table in crm_tables['table_name']:
    print(table)

print("Outlook Tables:")
for table in outlook_tables['table_name']:
    print(table)
    

CRM Tables:
accounts_alignment
employees
clients_contact
client_alignment
portfolio
fund_detail
Outlook Tables:
email_data
meeting_data


In [6]:
crm_engine = create_pg_engine('crm')
outlook_engine = create_pg_engine('outlook')

# Query columns for the 'crm' database
print("Columns in CRM Tables:")
for table in crm_tables['table_name']:
    print(f"\nTable: {table}")
    columns_query = f"""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = '{table}';
    """
    crm_columns = pd.read_sql(columns_query, crm_engine)
    
    for _, row in crm_columns.iterrows():
        print(f"  Column: {row['column_name']} | Type: {row['data_type']}")

# Query columns for the 'outlook' database
print("\nColumns in Outlook Tables:")
for table in outlook_tables['table_name']:
    print(f"\nTable: {table}")
    columns_query = f"""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = '{table}';
    """
    outlook_columns = pd.read_sql(columns_query, outlook_engine)
    
    for _, row in outlook_columns.iterrows():
        print(f"  Column: {row['column_name']} | Type: {row['data_type']}")

Columns in CRM Tables:

Table: accounts_alignment
  Column: employee_id | Type: text
  Column: employee_first_name | Type: text
  Column: employee_last_name | Type: text
  Column: account_id | Type: text
  Column: company | Type: text
  Column: industry | Type: text
  Column: contact_first_name | Type: text
  Column: contact_last_name | Type: text
  Column: contact_email | Type: text
  Column: contact_title | Type: text
  Column: contact_phone | Type: text

Table: employees
  Column: employee_id | Type: text
  Column: first_name | Type: text
  Column: last_name | Type: text
  Column: title | Type: text
  Column: email | Type: text
  Column: phone | Type: text
  Column: hire_date | Type: text
  Column: department | Type: text
  Column: market | Type: text

Table: clients_contact
  Column: company | Type: text
  Column: industry | Type: text
  Column: account_id | Type: text
  Column: contact_first_name | Type: text
  Column: contact_last_name | Type: text
  Column: contact_title | Type:

In [7]:
# For CRM Tables
print("First 10 Rows in CRM Tables:")
for table in crm_tables['table_name']:
    print(f"\n== Table: {table} ==")
    head_query = f"SELECT * FROM public.{table} LIMIT 10;"
    head_df = pd.read_sql(head_query, crm_engine)
    print(head_df)

# For Outlook Tables
print("\nFirst 10 Rows in Outlook Tables:")
for table in outlook_tables['table_name']:
    print(f"\n== Table: {table} ==")
    head_query = f"SELECT * FROM public.{table} LIMIT 10;"
    head_df = pd.read_sql(head_query, outlook_engine)
    print(head_df)

First 10 Rows in CRM Tables:

== Table: accounts_alignment ==
                            employee_id employee_first_name  \
0  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   
1  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   
2  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   
3  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   
4  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   
5  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   
6  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   
7  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   
8  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   
9  389300f0-5cac-4a60-ac26-e0d633027e4f                Jane   

  employee_last_name                            account_id  \
0         Moneypenny  f841a12f-43dc-44c5-953a-a78bb2050cbf   
1         Moneypenny  6c2395f7-24ff-4108-aef6-95f2be0a00dc   
2         Moneypenny  70185f63-e2b4-4105-afd5-7a6d1f6ea61b

#### Create SQLAlchemy engine

In [8]:
crm_outlook_engine = create_pg_engine('crm_outlook')

In [9]:
from sqlalchemy import text
# Clear all tables from the destination engine (crm_outlook_engine)
def clear_all_tables(dest_engine):

    tables_query = """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    AND table_type = 'BASE TABLE';
    """

    tables = pd.read_sql(tables_query, dest_engine)

    # Connect to the destination engine
    with dest_engine.connect() as conn:
        # Drop each table in the list
        for table in tables['table_name']:
            print(f"Dropping table: {table}")
            # Ensure the query is wrapped with text()
            drop_query = text(f'DROP TABLE IF EXISTS "{table}" CASCADE;')
            conn.execute(drop_query)  
    print("All existing tables dropped from crm_outlook.")

# Clear crm_outlook database
clear_all_tables(crm_outlook_engine)

Dropping table: accounts_alignment
Dropping table: employees
Dropping table: clients_contact
Dropping table: client_alignment
Dropping table: portfolio
Dropping table: fund_detail
Dropping table: email_data
Dropping table: meeting_data
All existing tables dropped from crm_outlook.


In [10]:
# Copy all tables from source engine to destination engine (crm_outlook)
def copy_all_tables(src_engine, dest_engine):

    tables_query = """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    AND table_type = 'BASE TABLE';
    """

    tables = pd.read_sql(tables_query, src_engine)


    for table in tables['table_name']:
        print(f"Copying table: {table}")
        # Read the table from the source database
        df = pd.read_sql(f'SELECT * FROM "{table}"', src_engine)
        # Write the table to the destination database
        df.to_sql(table, dest_engine, if_exists='replace', index=False)
        print(f"Table {table} copied successfully!")


# Copy from crm and outlook databases to crm_outlook
copy_all_tables(crm_engine, crm_outlook_engine)
copy_all_tables(outlook_engine, crm_outlook_engine)

print("All tables copied successfully to crm_outlook!")

Copying table: accounts_alignment
Table accounts_alignment copied successfully!
Copying table: employees
Table employees copied successfully!
Copying table: clients_contact
Table clients_contact copied successfully!
Copying table: client_alignment
Table client_alignment copied successfully!
Copying table: portfolio
Table portfolio copied successfully!
Copying table: fund_detail
Table fund_detail copied successfully!
Copying table: email_data
Table email_data copied successfully!
Copying table: meeting_data
Table meeting_data copied successfully!
All tables copied successfully to crm_outlook!


In [11]:
tables_query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables = pd.read_sql(tables_query, crm_outlook_engine)
print("Tables in crm_outlook:")
print(tables)


Tables in crm_outlook:
           table_name
0  accounts_alignment
1    client_alignment
2     clients_contact
3          email_data
4           employees
5         fund_detail
6        meeting_data
7           portfolio


In [12]:
meeting_query = "SELECT invitees, invitee_emails FROM meeting_data"
email_query = "SELECT to_names, to_emails FROM email_data"

meeting_df = pd.read_sql(meeting_query, crm_outlook_engine)
email_df = pd.read_sql(email_query, crm_outlook_engine)

# Print unique values for meeting_data
print("Unique values in 'meeting_data':")
print("\nColumn: invitees")
print(meeting_df['invitees'].dropna().unique())

print("\nColumn: invitee_emails")
print(meeting_df['invitee_emails'].dropna().unique())

# Print unique values for email_data
print("\nUnique values in 'email_data':")
print("\nColumn: to_names")
print(email_df['to_names'].dropna().unique())

print("\nColumn: to_emails")
print(email_df['to_emails'].dropna().unique())

Unique values in 'meeting_data':

Column: invitees
['Cynthia Hobbs' 'Jennifer Phelps' 'Kyle Waters' 'Denise Moore'
 'Adam Clay' 'Amy Winters' 'Roberto Martin' 'Jessica Palmer'
 'Michelle Jenkins' 'Ronnie Gray' 'David Moreno' 'Lisa Kennedy'
 'Mary Vasquez' 'Anna Lawrence' 'Tracey Smith' 'Kelly Smith'
 'Timothy Ochoa']

Column: invitee_emails
['cynthia.hobbs@abbvie.com' 'jennifer.phelps@aerovironment.com'
 'kyle.waters@amedisys.com' 'denise.moore@celestica.com'
 'adam.clay@compass.com' 'amy.winters@gamestopcorp.com'
 'roberto.martin@guardanthealth.com' 'jessica.palmer@hasbro.com'
 'michelle.jenkins@intuitivesurgical.com'
 'ronnie.gray@laddercapitalcorp.com' 'david.moreno@manpowergroup.com'
 'lisa.kennedy@lockheedmartincorporation.com' 'mary.vasquez@mariott.com'
 'anna.lawrence@matson.com' 'tracey.smith@medtronic.com'
 'kelly.smith@presidiopropertytrust.com' 'timothy.ochoa@hyatthotels.com']

Unique values in 'email_data':

Column: to_names
['Cynthia Hobbs' 'Jane Moneypenny' 'Jennifer Phel

In [16]:
# Query unique full names from clients_contact
clients_contact_query = """
SELECT DISTINCT contact_first_name || ' ' || contact_last_name AS full_name
FROM clients_contact
WHERE contact_first_name IS NOT NULL AND contact_last_name IS NOT NULL
"""

# Query unique full names from client_alignment where employee_last_name = 'Moneypenny'
client_alignment_query = """
SELECT DISTINCT contact_first_name || ' ' || contact_last_name AS full_name
FROM client_alignment
WHERE employee_last_name = 'Moneypenny'
  AND contact_first_name IS NOT NULL
  AND contact_last_name IS NOT NULL
"""

clients_contact_names = pd.read_sql(clients_contact_query, crm_outlook_engine)
moneypenny_names = pd.read_sql(client_alignment_query, crm_outlook_engine)


print("Unique full names from clients_contact:")
print(clients_contact_names['full_name'].unique())

print("\nUnique full names from client_alignment (Moneypenny only):")
print(moneypenny_names['full_name'].unique())

Unique full names from clients_contact:
['Sara Davis' 'Paul Powers' 'Adam Hobbs' 'Kathy Coffey' 'Robert Woods'
 'Tyler Mcclain' 'Sherry Garza' 'Kayla Daniels' 'Anna Lawrence'
 'Donald Smith' 'Kelly Smith' 'Anthony Simon' 'Rebecca Lee'
 'Sean Williams' 'Keith Mcknight' 'Justin Hernandez' 'Stephanie Haley'
 'Joseph Brewer' 'Patricia Woodward' 'David Moreno' 'Jennifer Warren'
 'Alexander Frank' 'Angelica Howard' 'Adam Johnson' 'Cynthia Hobbs'
 'Matthew Zavala' 'Michael Poole' 'Carrie Hill' 'Patrick Davis'
 'Ronnie Gray' 'Laura Wagner' 'Jessica Wilson' 'Joanne Banks'
 'Charles Stevens' 'Jessica Palmer' 'Paul Martin' 'Dawn Nielsen'
 'Susan Stewart' 'Tracy Stewart' 'Lisa Dunn' 'Rhonda Williams'
 'Paul Nunez' 'Michael Freeman' 'Michael Dominguez' 'Linda Goodman'
 'Gary Patel' 'Roberto Martin' 'Scott Velasquez' 'Kevin Thomas'
 'Andrea Mathews' 'Herbert Roman' 'Joshua Floyd' 'Joshua Jackson'
 'Tracey Smith' 'Margaret Brown' 'Pamela Hopkins' 'Edward Bauer'
 'Elizabeth Taylor' 'Kari Moreno' 'Leah

In [14]:
to_names = set(email_df['to_names'].dropna().unique())
clients_contact_overlap = to_names & set(
    pd.read_sql("""
        SELECT DISTINCT contact_first_name || ' ' || contact_last_name AS full_name
        FROM clients_contact
        WHERE contact_first_name IS NOT NULL AND contact_last_name IS NOT NULL
    """, crm_outlook_engine)['full_name']
)

moneypenny_overlap = to_names & set(
    pd.read_sql("""
        SELECT DISTINCT contact_first_name || ' ' || contact_last_name AS full_name
        FROM client_alignment
        WHERE employee_last_name = 'Moneypenny'
          AND contact_first_name IS NOT NULL
          AND contact_last_name IS NOT NULL
    """, crm_outlook_engine)['full_name']
)

# Print overlaps
print("Overlap with clients_contact:")
print(clients_contact_overlap)

print("Overlap with client_alignment (Moneypenny only):")
print(moneypenny_overlap)

Overlap with clients_contact:
{'Cynthia Hobbs', 'Kelly Smith', 'Jessica Palmer', 'Anna Lawrence', 'Kyle Waters', 'Ronnie Gray', 'Jennifer Phelps', 'Adam Clay', 'Roberto Martin', 'David Moreno', 'Denise Moore', 'Tracey Smith', 'Lisa Kennedy', 'Amy Winters', 'Michelle Jenkins', 'Timothy Ochoa', 'Mary Vasquez'}
Overlap with client_alignment (Moneypenny only):
{'Cynthia Hobbs', 'Kyle Waters', 'Jennifer Phelps', 'Roberto Martin', 'Adam Clay', 'Denise Moore', 'Tracey Smith', 'Lisa Kennedy', 'Michelle Jenkins', 'Mary Vasquez'}


## Inspect the Data

In [15]:
import datetime
import pytz

print(f"Notebook last execution time: {datetime.datetime.now(pytz.timezone('US/Central')).strftime('%a, %d %B %Y %H:%M:%S')}")

Notebook last execution time: Sun, 04 May 2025 22:06:00
